# JAX/MJX Dinosaur Training (Colab)

Train a dinosaur species using **MuJoCo MJX** (JAX-accelerated physics) with a
from-scratch PPO implementation in pure JAX. MJX vectorises thousands of parallel
simulations on a single GPU, giving 10-100x speedups over CPU-based Gymnasium training.

**Requirements:** Colab GPU runtime (A100 recommended).

**Supported Species:**
- `trex` — T-Rex: balance → locomotion → bite
- `velociraptor` — Raptor: balance → locomotion → strike
- `brachiosaurus` — Brachio: balance → locomotion → food reach

Set `SPECIES` in the configuration cell below to choose.

In [ ]:
# Install dependencies and verify GPU
!pip install mujoco mujoco-mjx "jax[cuda12]" flax optax

import os
import subprocess

if subprocess.run("nvidia-smi").returncode:
    raise RuntimeError("GPU not found. Use a GPU Colab runtime.")

NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
        f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

os.environ["MUJOCO_GL"] = "egl"

import jax
import mujoco
from mujoco import mjx

print(f"JAX devices: {jax.devices()}")
print(f"MuJoCo: {mujoco.__version__}")
print("Setup complete.")

In [ ]:
# Clone mesozoic-labs and install with JAX extras
!git clone https://github.com/kuds/mesozoic-labs.git /content/mesozoic-labs 2>/dev/null || echo 'Already cloned'
!pip install -e "/content/mesozoic-labs[jax]" -q

from IPython.display import clear_output

clear_output()
print("mesozoic-labs[jax] installed.")

In [ ]:
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import mujoco
import numpy as np
import optax
from mujoco import mjx

# Shared modules from mesozoic-labs
from environments.shared.reward_functions import (
    reward_forward_velocity,
    reward_alive,
    reward_energy,
    reward_posture,
    reward_approach_shaping,
    quat_to_tilt,
    check_height_tilt_termination,
)
from environments.shared.obs_functions import SensorLayout, build_bipedal_obs
from environments.shared.mjx_utils import scale_action_jax
from environments.shared.jax_ppo import (
    make_actor_critic,
    sample_action,
    compute_gae,
    ppo_loss,
    PPOConfig,
)
from environments.shared.jax_normalization import RunningMeanStd, normalize_obs, update_running_stats, decay_running_stats

In [ ]:
# ============================================================
# USER CONFIGURATION
# ============================================================
SPECIES = "trex"  # Choose: "trex", "velociraptor", "brachiosaurus"
CURRENT_STAGE = 1  # Curriculum stage: 1=balance, 2=locomotion, 3=species-specific
USE_GOOGLE_DRIVE = False  # Set to True to save outputs to Google Drive (persistent across sessions)
VERBOSE = 1  # 0=eval/summary only, 1=periodic updates (default), 2=every update

# Resume from a previous checkpoint (set to a .pkl path, or None to start fresh)
RESUME_FROM = None  # e.g. "trex_jax_checkpoint_100.pkl"

# ============================================================
# Species Configuration (auto-resolved from SPECIES)
# ============================================================
_SPECIES_CFG = {
    "trex": {
        "xml_path": "/content/mesozoic-labs/environments/trex/assets/trex.xml",
        "root_body": "pelvis",
        "healthy_z_range": (0.4, 1.6),
        "target_z_default": 0.5,
        "stage_names": {1: "Balance", 2: "Locomotion", 3: "Bite"},
    },
    "velociraptor": {
        "xml_path": "/content/mesozoic-labs/environments/velociraptor/assets/raptor.xml",
        "root_body": "pelvis",
        "healthy_z_range": (0.3, 1.0),
        "target_z_default": 0.3,
        "stage_names": {1: "Balance", 2: "Locomotion", 3: "Strike"},
    },
    "brachiosaurus": {
        "xml_path": "/content/mesozoic-labs/environments/brachiosaurus/assets/brachiosaurus.xml",
        "root_body": "torso",
        "healthy_z_range": (1.0, 3.5),
        "target_z_default": 3.0,
        "stage_names": {1: "Balance", 2: "Locomotion", 3: "Food Reach"},
    },
}

assert SPECIES in _SPECIES_CFG, f"Unknown species: {SPECIES}. Choose from: {list(_SPECIES_CFG)}"
_cfg = _SPECIES_CFG[SPECIES]

# ============================================================
# Storage Configuration (stage-organized directory structure)
# ============================================================
from datetime import datetime
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    _STORAGE_ROOT = Path("/content/drive/MyDrive/mesozoic-labs/logs/jax_training")
else:
    _STORAGE_ROOT = Path("logs")

_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = _STORAGE_ROOT / SPECIES / f"jax_{_timestamp}"
STAGE_DIR = RUN_DIR / f"stage{CURRENT_STAGE}"
MODEL_DIR = STAGE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# OUTPUT_DIR kept for backward compat (points to stage dir)
OUTPUT_DIR = STAGE_DIR

print(f"Species:    {SPECIES}")
print(f"Stage:      {CURRENT_STAGE} ({_cfg['stage_names'].get(CURRENT_STAGE, '?')})")
print(f"Run dir:    {RUN_DIR}")
print(f"Stage dir:  {STAGE_DIR}")
print(f"Drive:      {USE_GOOGLE_DRIVE}")
print(f"Verbose:    {VERBOSE}")
if RESUME_FROM:
    print(f"Resuming from checkpoint: {RESUME_FROM}")

## 1. Load Model into MJX

In [ ]:
# Load the MJCF model for the selected species
xml_path = _cfg["xml_path"]
mj_model = mujoco.MjModel.from_xml_path(xml_path)
mj_data = mujoco.MjData(mj_model)

print(f"{SPECIES.title()} model loaded:")
print(f"  Bodies: {mj_model.nbody}")
print(f"  Joints: {mj_model.njnt}")
print(f"  Actuators (nu): {mj_model.nu}")
print(f"  qpos dim: {mj_model.nq}")
print(f"  qvel dim: {mj_model.nv}")
print(f"  Total mass: {sum(mj_model.body_mass):.2f} kg")
print(f"  Timestep: {mj_model.opt.timestep * 1000:.1f} ms")

# Put model on device (GPU)
mjx_model = mjx.put_model(mj_model)
print(f"\nModel placed on {jax.devices()[0]}")

## 2. Environment Functions
Pure-JAX functions for observation, reward, reset, and step. These now use
the **shared modules** from `environments.shared` so that the same reward and
observation logic is used by both the Gymnasium (SB3) and MJX (JAX) training
paths.

**Note:** The shared `reward_functions` use `float()` / Python `if` which break
`jax.vmap` tracing. The reward and termination functions below are re-implemented
in pure JAX (`jnp` only) for vmap compatibility.

In [ ]:
# ---------- Constants ----------
FRAME_SKIP = 5
HEALTHY_Z_MIN, HEALTHY_Z_MAX = _cfg["healthy_z_range"]
MAX_TILT_ANGLE = 1.047
MAX_EPISODE_STEPS = 1000

# Body / geom / site IDs (looked up once from the MuJoCo model)
ROOT_BODY_ID = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, _cfg["root_body"])
FLOOR_GEOM_ID = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_GEOM, "floor")

# Sensor layout (matches MJCF): gyro(3), accel(3), quat(4), foot sensors...
# Bipedal species have 2 foot sensors at indices 10-11
# Quadrupedal (brachiosaurus) has 4 foot sensors at indices 10-13
if SPECIES == "brachiosaurus":
    SENSOR_LAYOUT = SensorLayout(gyro_start=0, accel_start=3, quat_start=6, foot_indices=(10, 11, 12, 13))
else:
    SENSOR_LAYOUT = SensorLayout(gyro_start=0, accel_start=3, quat_start=6, foot_indices=(10, 11))

# Action scaling ranges (as JAX arrays for use with shared scale_action_jax)
CTRL_RANGE = jnp.array(mj_model.actuator_ctrlrange)

print(f"Root body ({_cfg['root_body']}) id: {ROOT_BODY_ID}")
print(f"Action dim: {mj_model.nu}")

In [ ]:
def get_obs(data):
    """Extract observation vector from MJX data using shared obs builder."""
    return build_bipedal_obs(
        qpos=data.qpos,
        qvel=data.qvel,
        sensordata=data.sensordata,
        pelvis_xpos=data.xpos[ROOT_BODY_ID],
        target_pos=jnp.zeros(3),  # Target tracking handled by reward config
        sensor_layout=SENSOR_LAYOUT,
    )


# ---------- Pure-JAX reward & termination (vmap-safe) ----------
# The shared reward_functions use float()/min()/Python-if which break JAX
# tracing under jax.vmap. We re-implement the core math here with jnp only.

def _jax_quat_to_tilt(quat):
    """Tilt angle (radians) — pure JAX, no float() calls."""
    x, y = quat[1], quat[2]
    body_up_z = 1.0 - 2.0 * (x * x + y * y)
    return jnp.arccos(jnp.clip(body_up_z, -1.0, 1.0))


def compute_reward(data, action, reward_cfg):
    """Compute scalar reward using pure-JAX operations (vmap-safe)."""
    vel_2d = data.qvel[:2]
    forward_dir = jnp.array([1.0, 0.0])

    # Forward velocity reward
    forward_vel = jnp.dot(vel_2d, forward_dir)
    forward_vel_norm = jnp.clip(forward_vel / 8.0, -1.0, 1.0)
    r_forward = reward_cfg["forward_vel_weight"] * forward_vel_norm

    # Alive bonus
    r_alive = reward_cfg["alive_bonus"]

    # Energy penalty
    energy = jnp.sum(jnp.square(action)) / CTRL_RANGE.shape[0]
    r_energy = -reward_cfg["energy_penalty_weight"] * energy

    # Posture reward (quadratic tilt penalty)
    root_quat = data.sensordata[6:10]
    tilt = _jax_quat_to_tilt(root_quat)
    tilt_norm = jnp.minimum(tilt / MAX_TILT_ANGLE, 1.0)
    r_posture = -reward_cfg.get("posture_weight", 0.2) * (tilt_norm ** 2)

    return r_forward + r_alive + r_energy + r_posture


def is_terminated(data):
    """Check if the dinosaur has fallen — pure JAX, no Python control flow."""
    body_z = data.xpos[ROOT_BODY_ID, 2]
    root_quat = data.sensordata[6:10]
    tilt = _jax_quat_to_tilt(root_quat)
    fallen = body_z < HEALTHY_Z_MIN
    too_high = body_z > HEALTHY_Z_MAX
    tilted = tilt > MAX_TILT_ANGLE
    return fallen | too_high | tilted


def scale_action(action):
    """Scale action from [-1, 1] to actuator control range using shared function."""
    return scale_action_jax(action, CTRL_RANGE)


# Verify obs dimension
mujoco.mj_forward(mj_model, mj_data)
_test_data = mjx.put_data(mj_model, mj_data)
_test_obs = get_obs(_test_data)
OBS_DIM = _test_obs.shape[0]
ACT_DIM = mj_model.nu
print(f"Observation dim: {OBS_DIM}")
print(f"Action dim: {ACT_DIM}")

## 3. Batched MJX Step

A single `jax.jit`-compiled function that steps `N` parallel environments.

In [ ]:
def mjx_step_single(model, data, action):
    """Step one environment: apply action, advance physics, return new data."""
    ctrl = scale_action(action)
    data = data.replace(ctrl=ctrl)

    # Frame skip: step physics multiple times per action
    def body_fn(_, d):
        return mjx.step(model, d)

    data = jax.lax.fori_loop(0, FRAME_SKIP, body_fn, data)
    return data


# Vectorize: model is shared (None), data and action are batched (0)
@jax.jit
def batched_step(model, data_batch, action_batch):
    return jax.vmap(mjx_step_single, in_axes=(None, 0, 0))(model, data_batch, action_batch)


print("Batched step function compiled.")

## 4. Policy Network (Flax)
Uses the shared `ActorCritic` from `environments.shared.jax_ppo`.

In [ ]:
# Initialize using the shared ActorCritic from jax_ppo
network = make_actor_critic(action_dim=ACT_DIM)
rng = jax.random.PRNGKey(42)
dummy_obs = jnp.zeros((OBS_DIM,))
params = network.init(rng, dummy_obs)

# Observation normalization (stabilizes training across species/stages)
obs_rms = RunningMeanStd.create(OBS_DIM)

# Resume from checkpoint if specified
_resume_update = 0
_jax_kw_resume = {}
try:
    from environments.shared.config import load_stage_config as _lsc
    _jax_kw_resume = _lsc(SPECIES, CURRENT_STAGE).get("jax_kwargs", {})
except Exception:
    pass

if RESUME_FROM:
    import pickle
    _ckpt_path = OUTPUT_DIR / RESUME_FROM if not Path(RESUME_FROM).is_absolute() else Path(RESUME_FROM)
    with open(_ckpt_path, "rb") as f:
        _ckpt = pickle.load(f)
    params = jax.device_put(_ckpt["params"])
    _resume_update = _ckpt.get("update", 0)
    if "obs_rms" in _ckpt:
        obs_rms = _ckpt["obs_rms"]
        # When transitioning between stages, the obs distribution shifts
        # (e.g. near-zero velocity in balance → sustained velocity in
        # locomotion).  With 2048 envs the prior count can be ~65M, making
        # update_running_stats nearly a no-op.  Decay the count so new
        # data adapts the statistics within a few updates.
        _obs_decay = _jax_kw_resume.get("obs_rms_decay_on_resume", 0.01)
        if _obs_decay < 1.0:
            _old_count = obs_rms.count
            obs_rms = decay_running_stats(obs_rms, decay_factor=_obs_decay)
            print(f"  obs_rms count decayed: {_old_count:,.0f} → {obs_rms.count:,.0f} (factor={_obs_decay})")
    print(f"Resumed from {_ckpt_path} (update {_resume_update})")
    if "reward_history" in _ckpt:
        print(f"  Prior history: {len(_ckpt['reward_history'])} updates, best reward: {max(_ckpt['reward_history']):.4f}")

n_params = sum(p.size for p in jax.tree.leaves(params))
print(f"ActorCritic parameters: {n_params:,}")

## 5. PPO Implementation
Core PPO functions (`sample_action`, `compute_gae`, `ppo_loss`) are now
imported from `environments.shared.jax_ppo`. Notebook-specific wrappers
below adapt them for the training loop.

In [ ]:
# Use shared sample_action but with our network closure
def nb_sample_action(params, obs, rng):
    """Sample action from Gaussian policy using shared PPO module."""
    return sample_action(params, network, obs, rng)


# Use shared compute_gae directly (same signature)
# compute_gae is already imported from jax_ppo


# Notebook-specific PPO loss wrapper (uses the shared ppo_loss with extra metrics)
def nb_ppo_loss(params, obs, actions, old_log_probs, advantages, returns,
                clip_range=0.2, vf_coef=0.5, ent_coef=0.01):
    """PPO clipped surrogate loss using shared implementation."""
    mean, log_std, values = jax.vmap(network.apply, in_axes=(None, 0))(params, obs)
    std = jnp.exp(log_std)

    # Log probability of taken actions
    log_probs = -0.5 * jnp.sum(
        ((actions - mean) / (std + 1e-8)) ** 2 + 2 * log_std + jnp.log(2 * jnp.pi),
        axis=-1,
    )

    # Policy loss (clipped)
    ratio = jnp.exp(log_probs - old_log_probs)
    adv_normalized = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    surr1 = ratio * adv_normalized
    surr2 = jnp.clip(ratio, 1 - clip_range, 1 + clip_range) * adv_normalized
    policy_loss = -jnp.mean(jnp.minimum(surr1, surr2))

    # Value loss
    value_loss = jnp.mean((values - returns) ** 2)

    # Entropy bonus
    entropy = 0.5 * jnp.sum(jnp.log(2 * jnp.pi * jnp.e * std**2), axis=-1)
    entropy_bonus = jnp.mean(entropy)

    total_loss = policy_loss + vf_coef * value_loss - ent_coef * entropy_bonus

    aux = {
        "policy_loss": policy_loss,
        "value_loss": value_loss,
        "entropy": entropy_bonus,
        "approx_kl": jnp.mean((ratio - 1) - jnp.log(ratio)),
        "clip_fraction": jnp.mean((jnp.abs(ratio - 1.0) > clip_range).astype(jnp.float32)),
        "mean_std": jnp.mean(std),
    }

    return total_loss, aux


print("PPO functions defined (shared modules + notebook wrappers).")

## 6. Training Loop

In [ ]:
# ---------- Hyperparameters (loaded from TOML [jax] section) ----------
from environments.shared.config import load_stage_config

_stage_cfg = load_stage_config(SPECIES, CURRENT_STAGE)
_env_kw = _stage_cfg["env_kwargs"]
_jax_kw = _stage_cfg.get("jax_kwargs", {})

# JAX/MJX training hyperparameters (from TOML, with notebook defaults as fallback)
NUM_ENVS = _jax_kw.get("num_envs", 2048)
ROLLOUT_LEN = _jax_kw.get("rollout_len", 64)
NUM_UPDATES = _jax_kw.get("num_updates", 500)
PPO_EPOCHS = _jax_kw.get("ppo_epochs", 4)
MINIBATCH_SIZE = _jax_kw.get("minibatch_size", 512)
LEARNING_RATE = _jax_kw.get("learning_rate", 3e-4)
MAX_GRAD_NORM = _jax_kw.get("max_grad_norm", 0.5)
GAMMA = _jax_kw.get("gamma", 0.99)
GAE_LAMBDA = _jax_kw.get("gae_lambda", 0.95)
CLIP_RANGE = _jax_kw.get("clip_range", 0.2)
ENT_COEF = _jax_kw.get("ent_coef", 0.01)
FALL_PENALTY = _jax_kw.get("fall_penalty", -10.0)
RESET_NOISE_SCALE = _jax_kw.get("reset_noise_scale", 0.05)
INIT_QPOS_NOISE = _jax_kw.get("init_qpos_noise", 0.01)
INIT_YAW_NOISE = _jax_kw.get("init_yaw_noise", 0.1)

# Curriculum warmup (constrains policy updates while critic adapts to new reward landscape)
WARMUP_UPDATES = _jax_kw.get("warmup_updates", 0)          # 0 = no warmup (Stage 1 default)
WARMUP_CLIP_RANGE = _jax_kw.get("warmup_clip_range", 0.02)
WARMUP_ENT_COEF = _jax_kw.get("warmup_ent_coef", 0.02)

# Reward ramp (linearly ramp a reward weight from a fraction to its full value)
RAMP_UPDATES = _jax_kw.get("ramp_updates", 0)              # 0 = no ramp (Stage 1 default)
RAMP_ATTR = _jax_kw.get("ramp_attr", "forward_vel_weight")
RAMP_START_FRACTION = _jax_kw.get("ramp_start_fraction", 0.1)

# Reward config from [env] section
reward_cfg = {
    "forward_vel_weight": _env_kw.get("forward_vel_weight", 0.0),
    "alive_bonus": _env_kw.get("alive_bonus", 1.0),
    "energy_penalty_weight": _env_kw.get("energy_penalty_weight", 0.001),
    "posture_weight": _env_kw.get("posture_weight", 0.2),
}

print("Training config (from TOML [jax] section):")
print(f"  Species: {SPECIES}")
print(f"  Envs: {NUM_ENVS}")
print(f"  Rollout length: {ROLLOUT_LEN}")
print(f"  Updates: {NUM_UPDATES}")
print(f"  Total env steps: {NUM_ENVS * ROLLOUT_LEN * NUM_UPDATES:,}")
print(f"  Stage: {CURRENT_STAGE} ({_cfg['stage_names'].get(CURRENT_STAGE, '?')})")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Gamma: {GAMMA}")
print(f"  Clip range: {CLIP_RANGE}")
print(f"  Entropy coef: {ENT_COEF}")
print(f"  Max grad norm: {MAX_GRAD_NORM}")
print(f"  Fall penalty: {FALL_PENALTY}")
print(f"  Reset noise: joints={RESET_NOISE_SCALE}, xy={INIT_QPOS_NOISE}, yaw={INIT_YAW_NOISE}")
if WARMUP_UPDATES > 0:
    print(f"  Warmup: {WARMUP_UPDATES} updates (clip={WARMUP_CLIP_RANGE}, ent={WARMUP_ENT_COEF})")
if RAMP_UPDATES > 0:
    print(f"  Reward ramp: {RAMP_ATTR} from {RAMP_START_FRACTION:.0%} to 100% over {RAMP_UPDATES} updates")
print(f"  Reward config: {reward_cfg}")

In [ ]:
# Initialize batched environments
rng = jax.random.PRNGKey(42)

# Reset: create initial MJX data for all envs
mujoco.mj_resetData(mj_model, mj_data)
mujoco.mj_forward(mj_model, mj_data)
base_data = mjx.put_data(mj_model, mj_data)


# Replicate across batch with perturbations to position, orientation, and joints
def init_env(rng):
    rng_joint, rng_xy, rng_yaw = jax.random.split(rng, 3)

    # Joint angle perturbation
    joint_noise = jax.random.uniform(
        rng_joint, (mj_model.nq - 7,),
        minval=-RESET_NOISE_SCALE, maxval=RESET_NOISE_SCALE,
    )

    # XY position jitter (root position: qpos[0:2])
    xy_noise = jax.random.uniform(rng_xy, (2,), minval=-INIT_QPOS_NOISE, maxval=INIT_QPOS_NOISE)

    # Yaw rotation perturbation (rotate root quaternion around Z axis)
    yaw_angle = jax.random.uniform(rng_yaw, (), minval=-INIT_YAW_NOISE, maxval=INIT_YAW_NOISE)
    half_yaw = yaw_angle / 2.0
    # Yaw quaternion: [cos(yaw/2), 0, 0, sin(yaw/2)]
    yaw_quat = jnp.array([jnp.cos(half_yaw), 0.0, 0.0, jnp.sin(half_yaw)])
    # Multiply: yaw_quat * base_quat (Hamilton product)
    base_quat = base_data.qpos[3:7]
    w1, x1, y1, z1 = yaw_quat
    w2, x2, y2, z2 = base_quat
    new_quat = jnp.array([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2,
    ])

    qpos = base_data.qpos
    qpos = qpos.at[0:2].add(xy_noise)
    qpos = qpos.at[3:7].set(new_quat)
    qpos = qpos.at[7:].add(joint_noise)

    return base_data.replace(qpos=qpos)


rngs = jax.random.split(rng, NUM_ENVS)
env_batch = jax.vmap(init_env)(rngs)

# Forward pass to update derived quantities
env_batch = jax.jit(jax.vmap(mjx.forward, in_axes=(None, 0)))(mjx_model, env_batch)

print(f"Initialized {NUM_ENVS} parallel environments.")
print(f"qpos batch shape: {env_batch.qpos.shape}")
print(f"Reset noise: joints=±{RESET_NOISE_SCALE}, xy=±{INIT_QPOS_NOISE}m, yaw=±{INIT_YAW_NOISE}rad")

In [ ]:
# Optimizer (with gradient clipping)
optimizer = optax.chain(
    optax.clip_by_global_norm(MAX_GRAD_NORM),
    optax.adam(LEARNING_RATE),
)
opt_state = optimizer.init(params)


# JIT-compiled PPO update step (with gradient norm and loss decomposition)
@jax.jit
def ppo_update(params, opt_state, obs, actions, log_probs, advantages, returns,
               clip_range=CLIP_RANGE, ent_coef=ENT_COEF):
    (loss, aux), grads = jax.value_and_grad(nb_ppo_loss, has_aux=True)(
        params,
        obs,
        actions,
        log_probs,
        advantages,
        returns,
        clip_range=clip_range,
        ent_coef=ent_coef,
    )
    grad_norm = jnp.sqrt(sum(jnp.sum(g**2) for g in jax.tree.leaves(grads)))
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss, aux, grad_norm


# JIT-compiled batched action sampling
@jax.jit
def batched_sample(params, obs_batch, rng):
    rngs = jax.random.split(rng, obs_batch.shape[0])
    return jax.vmap(nb_sample_action, in_axes=(None, 0, 0))(params, obs_batch, rngs)


# JIT-compiled batched observation + reward
@jax.jit
def batched_obs(data_batch):
    return jax.vmap(get_obs)(data_batch)


@jax.jit
def batched_reward(data_batch, action_batch):
    return jax.vmap(compute_reward, in_axes=(0, 0, None))(data_batch, action_batch, reward_cfg)


@jax.jit
def batched_terminated(data_batch):
    return jax.vmap(is_terminated)(data_batch)


# Reset helper: reset fallen envs using a pre-computed fresh batch (avoids
# recomputing init_env + mjx.forward for ALL envs on every rollout step).
@jax.jit
def reset_fallen(env_batch, dones, fresh_batch):
    """Reset environments where done=True using pre-computed fresh states."""
    def select(fresh_field, existing_field):
        expand = dones.reshape((-1,) + (1,) * (existing_field.ndim - 1))
        return jnp.where(expand, fresh_field, existing_field)

    return jax.tree.map(select, fresh_batch, env_batch)


@jax.jit
def make_fresh_batch(rng):
    """Pre-compute a batch of fresh env states for resets."""
    rngs = jax.random.split(rng, NUM_ENVS)
    fresh = jax.vmap(init_env)(rngs)
    return jax.vmap(mjx.forward, in_axes=(None, 0))(mjx_model, fresh)


# ---------- JIT Warmup ----------
# Compile all JIT functions before the timing loop to avoid inflating SPS
# and to surface any compilation errors early.
print("Warming up JIT-compiled functions...")
_warmup_rng = jax.random.PRNGKey(0)
_warmup_obs = batched_obs(env_batch)
_warmup_actions, _warmup_lp, _warmup_vals = batched_sample(params, _warmup_obs, _warmup_rng)
_ = batched_step(mjx_model, env_batch, _warmup_actions)
_ = batched_reward(env_batch, _warmup_actions)
_ = batched_terminated(env_batch)
_ = make_fresh_batch(_warmup_rng)
jax.block_until_ready(_)
print(f"JIT warmup complete. Functions ready (grad clipping={MAX_GRAD_NORM}, obs normalization=on).")

In [ ]:
# ---------- Main Training Loop ----------
# Features: obs normalization (updated per-update, not per-step), bootstrap
# value for GAE, episode step tracking with MAX_EPISODE_STEPS truncation,
# proper episode return logging, gradient norm tracking, decomposed loss
# logging, periodic checkpointing with best-model tracking, CSV log, and
# ETA display.

import csv
import pickle

CHECKPOINT_FREQ = 25  # Save checkpoint every N updates
MAX_CHECKPOINTS = 5  # Keep only the last N checkpoints (saves disk space)

# Console log frequency based on VERBOSE:
# 0 = only final summary, 1 = every 20 updates (default), 2 = every update
_LOG_INTERVAL = {0: None, 1: 20, 2: 1}.get(VERBOSE, 20)

reward_history = []
loss_history = []
diagnostics_history = []  # Per-update detailed metrics
episode_return_history = []  # Completed episode returns (proper metric)

# Best model tracking
best_reward = -float("inf")
best_params = None
best_update = -1

# Episode return accumulators (per-env running totals)
_ep_returns = jnp.zeros(NUM_ENVS)
_ep_lengths = jnp.zeros(NUM_ENVS, dtype=jnp.int32)
_completed_returns = []  # Buffer for completed episode returns this update
_completed_lengths = []  # Buffer for completed episode lengths this update

# Checkpoint rotation list
_recent_checkpoints = []

# Warmup/ramp state
_warmup_active = WARMUP_UPDATES > 0
_ramp_active = RAMP_UPDATES > 0
_ramp_target_value = reward_cfg.get(RAMP_ATTR, 0.0) if _ramp_active else 0.0

if _warmup_active:
    # Store original values to restore after warmup
    _original_clip_range = CLIP_RANGE
    _original_ent_coef = ENT_COEF

# CSV log file for easy import into pandas/spreadsheets
csv_path = OUTPUT_DIR / f"{SPECIES}_jax_training_log.csv"
csv_fields = [
    "update",
    "reward_per_step",
    "episode_return",
    "episode_length",
    "total_loss",
    "policy_loss",
    "value_loss",
    "entropy",
    "approx_kl",
    "clip_fraction",
    "grad_norm",
    "mean_std",
    "steps",
    "sps",
    "fall_rate",
    "elapsed",
]
csv_file = open(csv_path, "w", newline="")
csv_writer = csv.DictWriter(csv_file, fieldnames=csv_fields)
csv_writer.writeheader()

_start_update = _resume_update
print(f"Starting training: updates {_start_update}..{_start_update + NUM_UPDATES - 1} "
      f"({ROLLOUT_LEN} steps x {NUM_ENVS} envs)")
print(f"Checkpoint frequency: every {CHECKPOINT_FREQ} updates (keep last {MAX_CHECKPOINTS})")
if _warmup_active:
    print(f"Warmup: updates 0..{WARMUP_UPDATES - 1} (clip_range={WARMUP_CLIP_RANGE}, ent_coef={WARMUP_ENT_COEF})")
if _ramp_active:
    print(f"Reward ramp: {RAMP_ATTR} from {_ramp_target_value * RAMP_START_FRACTION:.4f} to {_ramp_target_value:.4f} over updates 0..{RAMP_UPDATES - 1}")
print(f"CSV log: {csv_path}")
print("=" * 70)

t_start = time.time()

try:
    for update in range(_start_update, _start_update + NUM_UPDATES):
        # ---------- Pre-compute fresh batch for resets this update ----------
        rng, rng_fresh = jax.random.split(rng)
        fresh_batch = make_fresh_batch(rng_fresh)

        # ---------- Warmup: constrain policy during early updates ----------
        relative_update = update - _start_update
        if _warmup_active:
            if relative_update < WARMUP_UPDATES:
                _active_clip_range = WARMUP_CLIP_RANGE
                _active_ent_coef = WARMUP_ENT_COEF
            else:
                _active_clip_range = _original_clip_range
                _active_ent_coef = _original_ent_coef
                if relative_update == WARMUP_UPDATES and (_LOG_INTERVAL is not None):
                    print(f"  >>> Warmup complete at update {update}: restoring clip_range={_original_clip_range}, ent_coef={_original_ent_coef}")
        else:
            _active_clip_range = CLIP_RANGE
            _active_ent_coef = ENT_COEF

        # ---------- Reward ramp: linearly increase reward weight ----------
        if _ramp_active:
            if relative_update < RAMP_UPDATES:
                ramp_progress = relative_update / RAMP_UPDATES
                ramp_value = _ramp_target_value * (RAMP_START_FRACTION + (1.0 - RAMP_START_FRACTION) * ramp_progress)
            else:
                ramp_value = _ramp_target_value
            reward_cfg[RAMP_ATTR] = ramp_value

        # ---------- Collect rollout ----------
        all_obs, all_actions, all_log_probs, all_values = [], [], [], []
        all_rewards, all_dones = [], []

        for t in range(ROLLOUT_LEN):
            rng, rng_act = jax.random.split(rng)

            obs_raw = batched_obs(env_batch)

            # Normalize obs using current running stats (updated per-update below)
            obs = normalize_obs(obs_raw, obs_rms)

            actions, log_probs, values = batched_sample(params, obs, rng_act)

            # Step environments
            env_batch = batched_step(mjx_model, env_batch, actions)

            rewards = batched_reward(env_batch, actions)
            terminated = batched_terminated(env_batch)

            # Episode step tracking — truncate at MAX_EPISODE_STEPS
            _ep_lengths = _ep_lengths + 1
            truncated = _ep_lengths >= MAX_EPISODE_STEPS
            dones = terminated | truncated

            # Add fall penalty (only for termination, not truncation)
            rewards = rewards + terminated.astype(jnp.float32) * FALL_PENALTY

            # Track completed episode returns
            _ep_returns = _ep_returns + rewards
            done_mask = np.array(dones)
            if done_mask.any():
                _completed_returns.extend(np.array(_ep_returns)[done_mask].tolist())
                _completed_lengths.extend(np.array(_ep_lengths)[done_mask].tolist())

            all_obs.append(obs)
            all_actions.append(actions)
            all_log_probs.append(log_probs)
            all_values.append(values)
            all_rewards.append(rewards)
            all_dones.append(dones.astype(jnp.float32))

            # Reset done envs using pre-computed fresh batch and reset accumulators
            env_batch = reset_fallen(env_batch, dones, fresh_batch)
            _ep_returns = jnp.where(dones, 0.0, _ep_returns)
            _ep_lengths = jnp.where(dones, 0, _ep_lengths)

        # Stack rollout data: (T, NUM_ENVS, ...)
        obs_t = jnp.stack(all_obs)  # (T, N, obs_dim)
        act_t = jnp.stack(all_actions)  # (T, N, act_dim)
        lp_t = jnp.stack(all_log_probs)  # (T, N)
        val_t = jnp.stack(all_values)  # (T, N)
        rew_t = jnp.stack(all_rewards)  # (T, N)
        done_t = jnp.stack(all_dones)  # (T, N)

        # ---------- Update obs normalization stats (once per update) ----------
        obs_batch_flat = obs_t.reshape(-1, OBS_DIM)
        obs_rms = update_running_stats(obs_rms, obs_batch_flat)

        # ---------- Bootstrap value for GAE ----------
        # compute_gae expects values of shape (T+1, N) — append bootstrap
        rng, rng_bootstrap = jax.random.split(rng)
        obs_final_raw = batched_obs(env_batch)
        obs_final = normalize_obs(obs_final_raw, obs_rms)
        _, _, bootstrap_values = batched_sample(params, obs_final, rng_bootstrap)
        val_t_plus1 = jnp.concatenate([val_t, bootstrap_values[None]], axis=0)  # (T+1, N)

        # ---------- Compute advantages ----------
        advantages, returns = compute_gae(rew_t, val_t_plus1, done_t, GAMMA, GAE_LAMBDA)

        # Flatten: (T * N, ...)
        flat_obs = obs_t.reshape(-1, OBS_DIM)
        flat_act = act_t.reshape(-1, ACT_DIM)
        flat_lp = lp_t.reshape(-1)
        flat_adv = advantages.reshape(-1)
        flat_ret = returns.reshape(-1)

        # ---------- PPO update epochs ----------
        total_samples = flat_obs.shape[0]
        epoch_losses = []
        epoch_aux = []
        epoch_grad_norms = []

        for epoch in range(PPO_EPOCHS):
            rng, rng_perm = jax.random.split(rng)
            perm = jax.random.permutation(rng_perm, total_samples)

            for start in range(0, total_samples, MINIBATCH_SIZE):
                idx = perm[start : start + MINIBATCH_SIZE]
                mb_obs = flat_obs[idx]
                mb_act = flat_act[idx]
                mb_lp = flat_lp[idx]
                mb_adv = flat_adv[idx]
                mb_ret = flat_ret[idx]

                params, opt_state, loss, aux, grad_norm = ppo_update(
                    params,
                    opt_state,
                    mb_obs,
                    mb_act,
                    mb_lp,
                    mb_adv,
                    mb_ret,
                    clip_range=_active_clip_range,
                    ent_coef=_active_ent_coef,
                )
                epoch_losses.append(float(loss))
                epoch_aux.append({k: float(v) for k, v in aux.items()})
                epoch_grad_norms.append(float(grad_norm))

        avg_reward = float(rew_t.mean())
        avg_loss = np.mean(epoch_losses)
        avg_grad_norm = np.mean(epoch_grad_norms)
        avg_aux = {k: np.mean([a[k] for a in epoch_aux]) for k in epoch_aux[0]}
        fall_rate = float(done_t.sum()) / (ROLLOUT_LEN * NUM_ENVS)

        # Episode return stats (from completed episodes)
        if _completed_returns:
            mean_ep_return = np.mean(_completed_returns)
            mean_ep_length = np.mean(_completed_lengths)
        else:
            mean_ep_return = float("nan")
            mean_ep_length = float("nan")
        episode_return_history.append(mean_ep_return)

        reward_history.append(avg_reward)
        loss_history.append(avg_loss)
        diagnostics_history.append(
            {
                "reward": avg_reward,
                "episode_return": mean_ep_return,
                "episode_length": mean_ep_length,
                "loss": avg_loss,
                "grad_norm": avg_grad_norm,
                "fall_rate": fall_rate,
                **avg_aux,
            }
        )
        _completed_returns.clear()
        _completed_lengths.clear()

        # Track best model (use episode return when available, else per-step reward)
        _track_metric = mean_ep_return if not np.isnan(mean_ep_return) else avg_reward
        if _track_metric > best_reward:
            best_reward = _track_metric
            best_params = jax.device_get(params)
            best_update = update

        # Write to CSV log
        elapsed = time.time() - t_start
        steps_done = (update - _start_update + 1) * ROLLOUT_LEN * NUM_ENVS
        sps = steps_done / elapsed
        csv_writer.writerow(
            {
                "update": update,
                "reward_per_step": f"{avg_reward:.4f}",
                "episode_return": f"{mean_ep_return:.2f}" if not np.isnan(mean_ep_return) else "",
                "episode_length": f"{mean_ep_length:.1f}" if not np.isnan(mean_ep_length) else "",
                "total_loss": f"{avg_loss:.4f}",
                "policy_loss": f"{avg_aux['policy_loss']:.4f}",
                "value_loss": f"{avg_aux['value_loss']:.4f}",
                "entropy": f"{avg_aux['entropy']:.4f}",
                "approx_kl": f"{avg_aux['approx_kl']:.6f}",
                "clip_fraction": f"{avg_aux['clip_fraction']:.4f}",
                "grad_norm": f"{avg_grad_norm:.4f}",
                "mean_std": f"{avg_aux['mean_std']:.4f}",
                "steps": steps_done,
                "sps": f"{sps:.0f}",
                "fall_rate": f"{fall_rate:.4f}",
                "elapsed": f"{elapsed:.1f}",
            }
        )
        csv_file.flush()

        # Console logging (respects VERBOSE setting)
        if _LOG_INTERVAL is not None and (
            (update - _start_update) % _LOG_INTERVAL == 0
            or update == _start_update + NUM_UPDATES - 1
        ):
            updates_done = update - _start_update + 1
            updates_left = NUM_UPDATES - updates_done
            eta = (elapsed / updates_done) * updates_left if updates_done > 0 else 0
            eta_str = f"{eta / 60:.0f}m" if eta > 60 else f"{eta:.0f}s"
            ep_ret_str = f"ep_ret={mean_ep_return:+.1f}" if not np.isnan(mean_ep_return) else "ep_ret=n/a"
            print(
                f"[{update:4d}/{_start_update + NUM_UPDATES}]  "
                f"r/step={avg_reward:+.3f}  {ep_ret_str}  "
                f"loss={avg_loss:.4f}  "
                f"pi={avg_aux['policy_loss']:.3f}  v={avg_aux['value_loss']:.3f}  "
                f"ent={avg_aux['entropy']:.3f}  kl={avg_aux['approx_kl']:.4f}  "
                f"grad={avg_grad_norm:.3f}  falls={fall_rate:.1%}  "
                f"SPS={sps:,.0f}  ETA={eta_str}"
            )

        # Periodic checkpointing (with rotation)
        if (update - _start_update + 1) % CHECKPOINT_FREQ == 0:
            ckpt_path = MODEL_DIR / f"{SPECIES}_jax_checkpoint_{update + 1}.pkl"
            with open(ckpt_path, "wb") as f:
                pickle.dump(
                    {
                        "params": jax.device_get(params),
                        "obs_rms": obs_rms,
                        "update": update + 1,
                        "reward_history": reward_history,
                        "loss_history": loss_history,
                        "episode_return_history": episode_return_history,
                    },
                    f,
                )
            _recent_checkpoints.append(ckpt_path)
            # Remove old checkpoints beyond MAX_CHECKPOINTS
            while len(_recent_checkpoints) > MAX_CHECKPOINTS:
                old = _recent_checkpoints.pop(0)
                old.unlink(missing_ok=True)
            if _LOG_INTERVAL is not None:
                print(f"  >>> Checkpoint saved: {ckpt_path}")

finally:
    csv_file.close()

elapsed = time.time() - t_start
total_steps = NUM_UPDATES * ROLLOUT_LEN * NUM_ENVS
print("=" * 70)
print(f"Done! {total_steps:,} steps in {elapsed:.1f}s ({total_steps / elapsed:,.0f} SPS)")
print(f"Best metric: {best_reward:+.4f} at update {best_update}")

# Save final trained parameters and full training history
params_path = MODEL_DIR / f"{SPECIES}_jax_params.pkl"
with open(params_path, "wb") as f:
    pickle.dump(
        {
            "params": jax.device_get(params),
            "best_params": best_params,
            "best_reward": best_reward,
            "best_update": best_update,
            "obs_rms": obs_rms,
            "reward_history": reward_history,
            "loss_history": loss_history,
            "episode_return_history": episode_return_history,
            "diagnostics_history": diagnostics_history,
        },
        f,
    )
print(f"Parameters and training history saved to: {params_path}")
print(f"Training log CSV saved to: {csv_path}")

## Stage Gate Evaluation

Run full evaluation episodes on CPU to check whether the curriculum gate
thresholds (min reward and min episode length) have been met. Both conditions
must pass before advancing to the next training stage.

In [ ]:
# ---------- Stage Gate Evaluation ----------
# Run full evaluation episodes on CPU MuJoCo to measure proper episode
# rewards and lengths, then check against the curriculum gate thresholds
# from the TOML config. Collects biomechanical diagnostics for analysis.

from environments.shared.config import load_stage_config

N_EVAL_EPISODES = 25

# Load gate thresholds from TOML config
stage_config = load_stage_config(SPECIES, CURRENT_STAGE)
curriculum = stage_config.get("curriculum_kwargs", {})
gate_min_reward = curriculum.get("min_avg_reward", -float("inf"))
gate_min_length = curriculum.get("min_avg_episode_length", 0)

print(f"Stage {CURRENT_STAGE} curriculum gate thresholds:")
print(f"  min_avg_reward:         {gate_min_reward}")
print(f"  min_avg_episode_length: {gate_min_length}")
print(f"\nRunning {N_EVAL_EPISODES} evaluation episodes on CPU...")

# Use best params from training
eval_params = best_params if best_params is not None else jax.device_get(params)

eval_rewards = []
eval_lengths = []
eval_forward_vels = []
eval_distances = []
eval_tilt_angles = []
eval_pelvis_heights = []

# Per-step diagnostics (across all eval episodes, for plotting)
diag_tilt = []
diag_fwd_vel = []
diag_pelvis_h = []
diag_l_foot = []
diag_r_foot = []
diag_energy = []
diag_reward_components = {"forward": [], "alive": [], "energy": [], "posture": []}

for ep in range(N_EVAL_EPISODES):
    mujoco.mj_resetData(mj_model, mj_data)
    mj_data.qpos[7:] += np.random.uniform(-0.01, 0.01, size=mj_data.qpos[7:].shape)
    mujoco.mj_forward(mj_model, mj_data)

    ep_reward = 0.0
    ep_fwd_vels = []
    ep_tilts = []
    ep_heights = []
    start_pos = mj_data.qpos[:2].copy()

    for step in range(MAX_EPISODE_STEPS):
        cpu_data = mjx.put_data(mj_model, mj_data)
        obs_raw = get_obs(cpu_data)
        obs = normalize_obs(obs_raw, obs_rms)

        mean, log_std, value = network.apply(eval_params, obs)
        action = jnp.clip(mean, -1.0, 1.0)

        ctrl = np.array(scale_action(action))
        mj_data.ctrl[:] = ctrl
        for _ in range(FRAME_SKIP):
            mujoco.mj_step(mj_model, mj_data)

        # Collect per-step diagnostics
        fwd_vel = float(mj_data.qvel[0])
        body_z = float(mj_data.xpos[ROOT_BODY_ID, 2])
        root_quat = mj_data.sensordata[6:10]
        tilt = float(np.arccos(np.clip(1.0 - 2.0 * (root_quat[1]**2 + root_quat[2]**2), -1, 1)))
        energy = float(np.sum(np.square(np.array(action)))) / ACT_DIM

        ep_fwd_vels.append(fwd_vel)
        ep_tilts.append(tilt)
        ep_heights.append(body_z)

        diag_fwd_vel.append(fwd_vel)
        diag_tilt.append(tilt)
        diag_pelvis_h.append(body_z)
        diag_energy.append(energy)

        # Foot contact forces (from sensordata if available)
        if len(mj_data.sensordata) > 11:
            diag_l_foot.append(float(mj_data.sensordata[10]))
            diag_r_foot.append(float(mj_data.sensordata[11]))

        # Reward decomposition
        vel_2d = np.array(mj_data.qvel[:2])
        fwd_norm = float(np.clip(fwd_vel / 8.0, -1.0, 1.0))
        diag_reward_components["forward"].append(reward_cfg["forward_vel_weight"] * fwd_norm)
        diag_reward_components["alive"].append(reward_cfg["alive_bonus"])
        diag_reward_components["energy"].append(-reward_cfg["energy_penalty_weight"] * energy)
        tilt_norm = min(tilt / MAX_TILT_ANGLE, 1.0)
        diag_reward_components["posture"].append(-reward_cfg.get("posture_weight", 0.2) * tilt_norm**2)

        cpu_data = mjx.put_data(mj_model, mj_data)
        r = float(compute_reward(cpu_data, action, reward_cfg))
        ep_reward += r

        if body_z < HEALTHY_Z_MIN or body_z > HEALTHY_Z_MAX or tilt > MAX_TILT_ANGLE:
            break

    ep_length = step + 1
    distance = float(np.linalg.norm(mj_data.qpos[:2] - start_pos))

    eval_rewards.append(ep_reward)
    eval_lengths.append(ep_length)
    eval_forward_vels.append(np.mean(ep_fwd_vels))
    eval_distances.append(distance)
    eval_tilt_angles.append(np.mean(ep_tilts))
    eval_pelvis_heights.append(np.mean(ep_heights))

mean_reward = np.mean(eval_rewards)
mean_length = np.mean(eval_lengths)
std_reward = np.std(eval_rewards)
std_length = np.std(eval_lengths)
mean_fwd_vel = np.mean(eval_forward_vels)
std_fwd_vel = np.std(eval_forward_vels)
mean_distance = np.mean(eval_distances)
mean_tilt = np.mean(eval_tilt_angles)
mean_height = np.mean(eval_pelvis_heights)

print(f"\nEvaluation results ({N_EVAL_EPISODES} episodes):")
print(f"  Mean reward:      {mean_reward:.2f} +/- {std_reward:.2f}")
print(f"  Mean length:      {mean_length:.1f} +/- {std_length:.1f}")
print(f"  Mean fwd vel:     {mean_fwd_vel:.3f} +/- {std_fwd_vel:.3f} m/s")
print(f"  Mean distance:    {mean_distance:.2f} m")
print(f"  Mean tilt:        {np.degrees(mean_tilt):.1f} deg")
print(f"  Mean pelvis H:    {mean_height:.3f} m")

# Build stage results dict (compatible with reporting.save_results_csv)
stage_results = {
    "stage": CURRENT_STAGE,
    "name": _cfg["stage_names"].get(CURRENT_STAGE, f"Stage {CURRENT_STAGE}"),
    "description": f"JAX/MJX PPO stage {CURRENT_STAGE}",
    "timesteps": NUM_UPDATES * ROLLOUT_LEN * NUM_ENVS,
    "duration_seconds": elapsed,
    "mean_reward": round(mean_reward, 2),
    "std_reward": round(std_reward, 2),
    "mean_episode_length": round(mean_length, 1),
    "std_episode_length": round(std_length, 1),
    "mean_forward_vel": round(mean_fwd_vel, 3),
    "std_forward_vel": round(std_fwd_vel, 3),
    "mean_distance_traveled": round(mean_distance, 2),
    "best_eval_reward": round(best_reward, 2),
    "best_eval_timestep": best_update * ROLLOUT_LEN * NUM_ENVS,
    "sim_dt": mj_model.opt.timestep * FRAME_SKIP,
    "model_path": str(MODEL_DIR / "best_model"),
}

# Check gate conditions
gate_failures = []
if mean_reward < gate_min_reward:
    gate_failures.append(f"mean reward {mean_reward:.2f} < {gate_min_reward}")
if mean_length < gate_min_length:
    gate_failures.append(f"mean episode length {mean_length:.1f} < {gate_min_length}")

stage_results["gate_passed"] = len(gate_failures) == 0

if gate_failures:
    print(f"\n*** STAGE {CURRENT_STAGE} GATE NOT PASSED ***")
    for f in gate_failures:
        print(f"  - {f}")
    print("Do NOT proceed to the next stage. Re-train with more updates or adjusted hyperparameters.")
else:
    print(f"\n*** STAGE {CURRENT_STAGE} GATE PASSED ***")
    print("Safe to advance to the next stage.")

## Save Results & Stage Summary

Save structured training artifacts: stage summary text file, collected results
CSV (compatible with sweep analysis tooling), and model checkpoints.

In [ ]:
import csv
import json

# ============================================================
# 1. Stage summary text file (saved to stage dir)
# ============================================================
def _format_duration(seconds):
    """Format seconds into human-readable string."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    elif seconds < 3600:
        return f"{seconds / 60:.1f}m"
    else:
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        return f"{h}h {m}m"


stage_summary_path = STAGE_DIR / "stage_summary.txt"
sim_dt = stage_results.get("sim_dt", mj_model.opt.timestep * FRAME_SKIP)
_summary_lines = [
    f"Stage {CURRENT_STAGE}: {stage_results['name']}",
    "=" * 50,
    f"  Species:         {SPECIES}",
    f"  Algorithm:       JAX/MJX PPO",
    f"  Description:     {stage_results['description']}",
    f"  Timesteps:       {stage_results['timesteps']:,}",
    f"  Duration:        {_format_duration(stage_results['duration_seconds'])}",
    f"  Num envs:        {NUM_ENVS}",
    f"  Rollout len:     {ROLLOUT_LEN}",
    f"  PPO updates:     {NUM_UPDATES}",
    "",
    "Evaluation Results:",
    f"  Mean reward:     {stage_results['mean_reward']:.2f} +/- {stage_results['std_reward']:.2f}",
    f"  Mean ep length:  {stage_results['mean_episode_length']:.1f} +/- {stage_results['std_episode_length']:.1f} steps "
    f"({stage_results['mean_episode_length'] * sim_dt:.2f}s sim time)",
    f"  Mean fwd vel:    {stage_results['mean_forward_vel']:.3f} +/- {stage_results['std_forward_vel']:.3f} m/s",
    f"  Mean distance:   {stage_results['mean_distance_traveled']:.2f} m",
    f"  Best eval:       {stage_results['best_eval_reward']:.2f} (at {stage_results['best_eval_timestep']:,} steps)",
    "",
    f"Gate passed:       {stage_results['gate_passed']}",
    f"Model path:        {stage_results['model_path']}",
]
stage_summary_text = "\n".join(_summary_lines) + "\n"
stage_summary_path.write_text(stage_summary_text)
print(f"Stage summary saved: {stage_summary_path}")
print()
print(stage_summary_text)

# ============================================================
# 2. Save stage config snapshot
# ============================================================
config_snapshot = {
    "species": SPECIES,
    "stage": CURRENT_STAGE,
    "algorithm": "jax_ppo",
    "jax_kwargs": {
        "num_envs": NUM_ENVS,
        "rollout_len": ROLLOUT_LEN,
        "num_updates": NUM_UPDATES,
        "ppo_epochs": PPO_EPOCHS,
        "minibatch_size": MINIBATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "max_grad_norm": MAX_GRAD_NORM,
        "gamma": GAMMA,
        "gae_lambda": GAE_LAMBDA,
        "clip_range": CLIP_RANGE,
        "ent_coef": ENT_COEF,
        "fall_penalty": FALL_PENALTY,
        "reset_noise_scale": RESET_NOISE_SCALE,
        "init_qpos_noise": INIT_QPOS_NOISE,
        "init_yaw_noise": INIT_YAW_NOISE,
        "warmup_updates": WARMUP_UPDATES,
        "warmup_clip_range": WARMUP_CLIP_RANGE,
        "warmup_ent_coef": WARMUP_ENT_COEF,
        "ramp_updates": RAMP_UPDATES,
        "ramp_attr": RAMP_ATTR,
        "ramp_start_fraction": RAMP_START_FRACTION,
    },
    "reward_cfg": reward_cfg,
    "env_kwargs": dict(_env_kw),
    "curriculum_kwargs": dict(_stage_cfg.get("curriculum_kwargs", {})),
}
config_path = STAGE_DIR / "stage_config.json"
with open(config_path, "w") as f:
    json.dump(config_snapshot, f, indent=2)
print(f"Stage config saved: {config_path}")

# ============================================================
# 3. Collected results CSV (compatible with sweep tooling)
# ============================================================
csv_results_path = RUN_DIR / "collected_results.csv"
_csv_existed = csv_results_path.exists()

# Read existing rows if appending to multi-stage run
_existing_rows = []
if _csv_existed:
    with open(csv_results_path, "r") as f:
        reader = csv.DictReader(f)
        _existing_rows = [row for row in reader if int(row.get("stage", 0)) != CURRENT_STAGE]

_result_row = {
    "species": SPECIES,
    "algorithm": "jax_ppo",
    "seed": 42,
    "stage": CURRENT_STAGE,
    "best_mean_reward": stage_results["best_eval_reward"],
    "last_mean_reward": stage_results["mean_reward"],
    "last_mean_episode_length": stage_results["mean_episode_length"],
    "mean_forward_vel": stage_results["mean_forward_vel"],
    "std_forward_vel": stage_results["std_forward_vel"],
    "mean_distance_traveled": stage_results["mean_distance_traveled"],
    "training_duration_seconds": round(stage_results["duration_seconds"], 1),
    "reward_threshold": curriculum.get("min_avg_reward", ""),
    "ep_length_threshold": curriculum.get("min_avg_episode_length", ""),
    "stage_passed": stage_results["gate_passed"],
}

_all_rows = _existing_rows + [_result_row]
_all_fieldnames = list(dict.fromkeys(
    k for row in _all_rows for k in row.keys()
))
with open(csv_results_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=_all_fieldnames)
    writer.writeheader()
    writer.writerows(_all_rows)
print(f"Collected results CSV saved: {csv_results_path}")

# ============================================================
# 4. Save diagnostics.npz (for downstream visualization tooling)
# ============================================================
_diag_data = {
    "tilt_angle": np.array(diag_tilt),
    "forward_vel": np.array(diag_fwd_vel),
    "pelvis_height": np.array(diag_pelvis_h),
    "energy": np.array(diag_energy),
}
if diag_l_foot:
    _diag_data["l_foot_contact"] = np.array(diag_l_foot)
    _diag_data["r_foot_contact"] = np.array(diag_r_foot)
for comp_name, comp_vals in diag_reward_components.items():
    _diag_data[f"reward_{comp_name}"] = np.array(comp_vals)

np.savez(STAGE_DIR / "diagnostics.npz", **_diag_data)
print(f"Diagnostics saved: {STAGE_DIR / 'diagnostics.npz'}")

# ============================================================
# 5. Save best model params
# ============================================================
import pickle

best_model_path = MODEL_DIR / "best_model.pkl"
with open(best_model_path, "wb") as f:
    pickle.dump(
        {
            "params": best_params if best_params is not None else jax.device_get(params),
            "obs_rms": obs_rms,
            "best_reward": best_reward,
            "best_update": best_update,
        },
        f,
    )
final_model_path = MODEL_DIR / f"stage{CURRENT_STAGE}_final.pkl"
with open(final_model_path, "wb") as f:
    pickle.dump(
        {
            "params": jax.device_get(params),
            "obs_rms": obs_rms,
        },
        f,
    )
print(f"Best model saved:  {best_model_path}")
print(f"Final model saved: {final_model_path}")

# ============================================================
# 6. Full training summary text file (run-level)
# ============================================================
training_summary_path = RUN_DIR / "training_summary.txt"
_ts_lines = [
    "Mesozoic Labs JAX/MJX Training Summary",
    "=" * 50,
    "",
    f"Species:        {SPECIES.title()}",
    f"Algorithm:      JAX/MJX PPO",
    f"Date:           {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"Seed:           42",
    f"Parallel envs:  {NUM_ENVS}",
    f"Run directory:  {RUN_DIR}",
    "",
    f"Stage {stage_results['stage']}: {stage_results['name']}",
    f"  Description:    {stage_results['description']}",
    f"  Timesteps:      {stage_results['timesteps']:,}",
    f"  Duration:       {_format_duration(stage_results['duration_seconds'])}",
    f"  Final eval:     {stage_results['mean_reward']:.2f} +/- {stage_results['std_reward']:.2f}",
    f"  Avg ep length:  {stage_results['mean_episode_length']:.1f} +/- {stage_results['std_episode_length']:.1f} steps "
    f"({stage_results['mean_episode_length'] * sim_dt:.2f}s sim time)",
    f"  Avg fwd vel:    {stage_results['mean_forward_vel']:.3f} +/- {stage_results['std_forward_vel']:.3f} m/s",
    f"  Best eval:      {stage_results['best_eval_reward']:.2f} (at {stage_results['best_eval_timestep']:,} steps)",
    f"  Best model:     {best_model_path}",
    f"  Gate passed:    {stage_results['gate_passed']}",
    "",
    "-" * 50,
    f"Total training time: {_format_duration(stage_results['duration_seconds'])}",
]
training_summary_text = "\n".join(_ts_lines) + "\n"
training_summary_path.write_text(training_summary_text)
print(f"\nTraining summary saved: {training_summary_path}")

## 7. Training Curves & Locomotion Diagnostics

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
window = min(20, len(reward_history) // 4 + 1)

# Reward per step
ax = axes[0, 0]
ax.plot(reward_history, "b-", alpha=0.3, label="per-update")
if window > 1:
    smoothed = np.convolve(reward_history, np.ones(window) / window, mode="valid")
    ax.plot(range(window - 1, len(reward_history)), smoothed, "b-", linewidth=2, label=f"{window}-update avg")
ax.set_xlabel("PPO Update")
ax.set_ylabel("Mean Reward/Step")
ax.set_title("Reward per Step")
ax.legend()
ax.grid(True, alpha=0.3)

# Episode return (the proper RL metric)
ax = axes[0, 1]
valid_returns = [(i, r) for i, r in enumerate(episode_return_history) if not np.isnan(r)]
if valid_returns:
    idxs, vals = zip(*valid_returns)
    ax.plot(idxs, vals, "g-", alpha=0.3, label="per-update")
    if window > 1 and len(vals) >= window:
        smoothed_ret = np.convolve(vals, np.ones(window) / window, mode="valid")
        ax.plot(range(idxs[0] + window - 1, idxs[0] + window - 1 + len(smoothed_ret)),
                smoothed_ret, "g-", linewidth=2, label=f"{window}-update avg")
ax.set_xlabel("PPO Update")
ax.set_ylabel("Episode Return")
ax.set_title("Episode Return")
ax.legend()
ax.grid(True, alpha=0.3)

# Total loss
ax = axes[0, 2]
ax.plot(loss_history, "r-", alpha=0.5)
ax.set_xlabel("PPO Update")
ax.set_ylabel("Loss")
ax.set_title("Total Loss")
ax.grid(True, alpha=0.3)

# Gradient norm
ax = axes[1, 0]
grad_norms = [d["grad_norm"] for d in diagnostics_history]
ax.plot(grad_norms, "g-", alpha=0.5)
ax.set_xlabel("PPO Update")
ax.set_ylabel("Gradient Norm")
ax.set_title("Gradient Norm (pre-clip)")
ax.grid(True, alpha=0.3)

# Policy loss vs Value loss
ax = axes[1, 1]
pi_losses = [d["policy_loss"] for d in diagnostics_history]
v_losses = [d["value_loss"] for d in diagnostics_history]
ax.plot(pi_losses, "b-", alpha=0.5, label="Policy loss")
ax.plot(v_losses, "r-", alpha=0.5, label="Value loss")
ax.set_xlabel("PPO Update")
ax.set_ylabel("Loss")
ax.set_title("Policy vs Value Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# Entropy and KL divergence
ax = axes[1, 2]
entropies = [d["entropy"] for d in diagnostics_history]
ax.plot(entropies, "purple", alpha=0.7, label="Entropy")
ax.set_xlabel("PPO Update")
ax.set_ylabel("Entropy")
ax.set_title("Policy Entropy")
ax2 = ax.twinx()
kls = [d["approx_kl"] for d in diagnostics_history]
ax2.plot(kls, "orange", alpha=0.7, label="Approx KL")
ax2.set_ylabel("Approx KL")
ax.legend(loc="upper left")
ax2.legend(loc="upper right")
ax.grid(True, alpha=0.3)

# Fall rate
ax = axes[2, 0]
fall_rates = [d["fall_rate"] for d in diagnostics_history]
ax.plot(fall_rates, "brown", alpha=0.5)
if window > 1:
    smoothed_falls = np.convolve(fall_rates, np.ones(window) / window, mode="valid")
    ax.plot(range(window - 1, len(fall_rates)), smoothed_falls, "brown", linewidth=2)
ax.set_xlabel("PPO Update")
ax.set_ylabel("Fall Rate")
ax.set_title("Episode Termination Rate")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# Episode length
ax = axes[2, 1]
valid_lengths = [(i, d["episode_length"]) for i, d in enumerate(diagnostics_history)
                 if not np.isnan(d.get("episode_length", float("nan")))]
if valid_lengths:
    idxs_l, vals_l = zip(*valid_lengths)
    ax.plot(idxs_l, vals_l, "teal", alpha=0.5)
    if window > 1 and len(vals_l) >= window:
        smoothed_len = np.convolve(vals_l, np.ones(window) / window, mode="valid")
        ax.plot(range(idxs_l[0] + window - 1, idxs_l[0] + window - 1 + len(smoothed_len)),
                smoothed_len, "teal", linewidth=2)
ax.set_xlabel("PPO Update")
ax.set_ylabel("Steps")
ax.set_title("Mean Episode Length")
ax.grid(True, alpha=0.3)

# Clip fraction and mean std
ax = axes[2, 2]
clip_fracs = [d["clip_fraction"] for d in diagnostics_history]
mean_stds = [d["mean_std"] for d in diagnostics_history]
ax.plot(clip_fracs, "red", alpha=0.5, label="Clip fraction")
ax.set_ylabel("Clip Fraction")
ax.set_xlabel("PPO Update")
ax3 = ax.twinx()
ax3.plot(mean_stds, "blue", alpha=0.5, label="Mean std")
ax3.set_ylabel("Mean Std")
ax.legend(loc="upper left")
ax3.legend(loc="upper right")
ax.set_title("Clip Fraction & Policy Std")
ax.grid(True, alpha=0.3)

plt.suptitle(f"{SPECIES.title()} JAX/MJX Training Diagnostics (Stage {CURRENT_STAGE})", fontsize=14, fontweight="bold")
plt.tight_layout()
curve_path = OUTPUT_DIR / f"{SPECIES}_jax_training_curves.png"
plt.savefig(curve_path, dpi=150)
print(f"Training curves saved to: {curve_path}")
plt.show()

In [ ]:
# ---------- Locomotion Diagnostics ----------
# Biomechanical analysis from the evaluation episodes, matching the diagnostic
# plots generated by training.ipynb (locomotion_health.png, foot_contacts.png).

def _smooth(values, window=50):
    """Rolling-mean smoothing filter."""
    values = np.asarray(values, dtype=float)
    if len(values) < window:
        return values
    kernel = np.ones(window) / window
    smoothed = np.convolve(values, kernel, mode="same")
    hw = window // 2
    smoothed[:hw] = values[:hw]
    smoothed[-hw:] = values[-hw:]
    return smoothed

fig_diag, axes_d = plt.subplots(2, 3, figsize=(18, 10))
_steps = np.arange(len(diag_tilt))

# Tilt angle (degrees)
ax = axes_d[0, 0]
ax.plot(_steps, _smooth(np.degrees(diag_tilt)), "b-", alpha=0.8)
ax.axhline(np.degrees(MAX_TILT_ANGLE), color="red", linestyle="--", alpha=0.5, label="Max tilt")
ax.set_xlabel("Eval Step")
ax.set_ylabel("Tilt (degrees)")
ax.set_title("Tilt Angle")
ax.legend()
ax.grid(True, alpha=0.3)

# Forward velocity
ax = axes_d[0, 1]
ax.plot(_steps, _smooth(diag_fwd_vel), "g-", alpha=0.8)
ax.axhline(0, color="gray", linestyle="--", alpha=0.3)
ax.set_xlabel("Eval Step")
ax.set_ylabel("Forward Vel (m/s)")
ax.set_title("Forward Velocity")
ax.grid(True, alpha=0.3)

# Pelvis height
ax = axes_d[0, 2]
ax.plot(_steps, _smooth(diag_pelvis_h), "teal", alpha=0.8)
ax.axhline(HEALTHY_Z_MIN, color="red", linestyle="--", alpha=0.5, label="Z min")
ax.axhline(HEALTHY_Z_MAX, color="red", linestyle="--", alpha=0.5, label="Z max")
ax.set_xlabel("Eval Step")
ax.set_ylabel("Height (m)")
ax.set_title("Pelvis Height")
ax.legend()
ax.grid(True, alpha=0.3)

# Foot contacts (if available)
ax = axes_d[1, 0]
if diag_l_foot and diag_r_foot:
    _foot_steps = np.arange(len(diag_l_foot))
    ax.plot(_foot_steps, _smooth(diag_l_foot), "b-", alpha=0.6, label="Left foot")
    ax.plot(_foot_steps, _smooth(diag_r_foot), "r-", alpha=0.6, label="Right foot")
    # Gait symmetry
    _l = np.array(diag_l_foot)
    _r = np.array(diag_r_foot)
    _sym = 1.0 - np.abs(_l - _r) / (_l + _r + 1e-8)
    ax.plot(_foot_steps, _smooth(_sym), "purple", alpha=0.5, linestyle="--", label="Gait symmetry")
    ax.legend()
    ax.set_xlabel("Eval Step")
    ax.set_ylabel("Contact Force / Symmetry")
    ax.set_title("Foot Contacts & Gait Symmetry")
else:
    ax.text(0.5, 0.5, "No foot contact data", transform=ax.transAxes, ha="center", va="center")
    ax.set_title("Foot Contacts (N/A)")
ax.grid(True, alpha=0.3)

# Reward decomposition
ax = axes_d[1, 1]
_comp_steps = np.arange(len(diag_reward_components["forward"]))
for comp_name, comp_vals in diag_reward_components.items():
    ax.plot(_comp_steps, _smooth(comp_vals), alpha=0.7, label=comp_name)
ax.set_xlabel("Eval Step")
ax.set_ylabel("Reward Component")
ax.set_title("Reward Decomposition")
ax.legend()
ax.grid(True, alpha=0.3)

# Cost of transport (energy / forward velocity)
ax = axes_d[1, 2]
_energy = np.abs(np.array(diag_energy))
_fwd = np.maximum(np.array(diag_fwd_vel), 0.01)
_cot = _energy / _fwd
ax.plot(_steps, _smooth(_cot), "brown", alpha=0.8)
ax.set_xlabel("Eval Step")
ax.set_ylabel("Cost of Transport")
ax.set_title("Cost of Transport")
ax.grid(True, alpha=0.3)

plt.suptitle(f"{SPECIES.title()} Stage {CURRENT_STAGE} — Locomotion Diagnostics", fontsize=14, fontweight="bold")
plt.tight_layout()

diag_plot_path = STAGE_DIR / "locomotion_health.png"
plt.savefig(diag_plot_path, dpi=150)
print(f"Locomotion diagnostics saved: {diag_plot_path}")
plt.show()

# Separate foot contact detail plot (matches training.ipynb foot_contacts.png)
if diag_l_foot and diag_r_foot:
    fig_foot, axes_f = plt.subplots(1, 3, figsize=(15, 4))
    _foot_steps = np.arange(len(diag_l_foot))

    ax = axes_f[0]
    ax.plot(_foot_steps, _smooth(diag_l_foot, window=20), "b-", alpha=0.7, label="Left")
    ax.plot(_foot_steps, _smooth(diag_r_foot, window=20), "r-", alpha=0.7, label="Right")
    ax.set_title("Raw Contact Forces")
    ax.set_xlabel("Eval Step")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes_f[1]
    _stride_proxy = (np.array(diag_l_foot) + np.array(diag_r_foot)) / 2.0
    ax.plot(_foot_steps, _smooth(_stride_proxy, window=20), "green", alpha=0.7)
    ax.set_title("Stride Frequency Proxy")
    ax.set_xlabel("Eval Step")
    ax.grid(True, alpha=0.3)

    ax = axes_f[2]
    ax.plot(_foot_steps, _smooth(_sym, window=20), "purple", alpha=0.7)
    ax.set_ylim(0, 1.1)
    ax.set_title("Gait Symmetry")
    ax.set_xlabel("Eval Step")
    ax.grid(True, alpha=0.3)

    plt.suptitle(f"{SPECIES.title()} — Foot Contact Analysis", fontsize=13, fontweight="bold")
    plt.tight_layout()
    foot_path = STAGE_DIR / "foot_contacts.png"
    plt.savefig(foot_path, dpi=150)
    print(f"Foot contacts saved: {foot_path}")
    plt.show()

## 8. Record Training Video

Record a video of the best trained policy (highest reward during training) using
the CPU MuJoCo renderer. The JAX policy is evaluated deterministically (using the
action mean).

In [ ]:
try:
    import mediapy

    _HAS_MEDIAPY = True
except ImportError:
    _HAS_MEDIAPY = False
    print("mediapy not installed. Install with: pip install mediapy")

if _HAS_MEDIAPY:
    # Use the best model (highest reward during training) for video recording
    video_params = best_params if best_params is not None else jax.device_get(params)
    print(f"Recording video with best model (update {best_update}, reward {best_reward:+.4f})")

    # Set up CPU-based MuJoCo renderer
    renderer = mujoco.Renderer(mj_model, height=480, width=640)

    # Reset environment
    mujoco.mj_resetData(mj_model, mj_data)
    mujoco.mj_forward(mj_model, mj_data)

    frames = []
    episode_reward = 0.0

    for step in range(MAX_EPISODE_STEPS):
        # Get observation from CPU data
        cpu_data = mjx.put_data(mj_model, mj_data)
        obs_raw = get_obs(cpu_data)
        obs = normalize_obs(obs_raw, obs_rms)

        # Get deterministic action (use mean, no noise)
        mean, log_std, value = network.apply(video_params, obs)
        action = jnp.clip(mean, -1.0, 1.0)

        # Apply action to CPU sim
        ctrl = np.array(scale_action(action))
        mj_data.ctrl[:] = ctrl
        for _ in range(FRAME_SKIP):
            mujoco.mj_step(mj_model, mj_data)

        # Render frame
        renderer.update_scene(mj_data)
        frames.append(renderer.render())

        # Compute reward
        cpu_data = mjx.put_data(mj_model, mj_data)
        r = float(compute_reward(cpu_data, action, reward_cfg))
        episode_reward += r

        # Check termination
        body_z = mj_data.xpos[ROOT_BODY_ID, 2]
        if body_z < HEALTHY_Z_MIN or body_z > HEALTHY_Z_MAX:
            break

    renderer.close()

    video_path = str(OUTPUT_DIR / f"{SPECIES}_jax_mjx_training.mp4")
    mediapy.write_video(video_path, frames, fps=50)
    print(f"Episode reward: {episode_reward:.2f} | {len(frames)} frames")
    print(f"Saved to: {video_path}")
    mediapy.show_video(frames, fps=50)

## 9. Next Steps

To continue with curriculum training, change `CURRENT_STAGE` in the configuration
cell above and re-run from there. The reward config is loaded automatically from the
TOML files in `configs/<species>/`:

```python
CURRENT_STAGE = 2  # or 3
```

The policy parameters carry over automatically between stages.
After each stage, re-run the Stage Gate Evaluation cell to verify the curriculum
thresholds are met, then re-run the video recording cell to capture the new behavior.

**Resuming from a checkpoint:** If your Colab session disconnects, set
`RESUME_FROM` in the configuration cell to reload parameters and obs stats:

```python
RESUME_FROM = "trex_jax_checkpoint_100.pkl"
```

To train a different species, change `SPECIES` in the configuration cell and
restart from the beginning.

## 10. Auto-Disconnect (Optional)

Optionally disconnect the Colab runtime after completion to free up resources.
Set `AUTO_DISCONNECT = True` in the configuration cell or toggle below.

In [ ]:
AUTO_DISCONNECT = True  # Set to True to disconnect runtime after training

if AUTO_DISCONNECT:
    import time
    print("Training finished. Disconnecting runtime in 5 seconds...")
    time.sleep(5)
    from google.colab import runtime
    runtime.unassign()
else:
    print("Training finished. Runtime kept alive — remember to disconnect manually when done.")